# SitePulse — Data Cleaning & Validation

## Objective

This notebook evaluates the SitePulse data cleaning pipeline by comparing the raw operational dataset with the cleaned dataset.

The cleaning process addresses:

- duplicate records
- missing workforce values
- missing fuel consumption
- missing material delivery values
- missing weather information
- incorrect date data types
- core business-rule validation

The objective is to verify that the processed dataset is complete, consistent, and ready for exploratory analysis and feature engineering.

In [2]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")

Project root: /Users/beyzatapan/Desktop/sitepulse-construction-analytics
Raw data directory: /Users/beyzatapan/Desktop/sitepulse-construction-analytics/data/raw
Processed data directory: /Users/beyzatapan/Desktop/sitepulse-construction-analytics/data/processed


In [4]:
raw = pd.read_csv(
    RAW_DATA_DIR / "daily_operations.csv"
)

cleaned = pd.read_csv(
    PROCESSED_DATA_DIR / "daily_operations_clean.csv",
    parse_dates=["date"],
)

In [5]:
print("Raw shape:", raw.shape)
print("Cleaned shape:", cleaned.shape)

Raw shape: (1396, 15)
Cleaned shape: (1390, 15)


In [6]:
quality_comparison = pd.DataFrame(
    {
        "metric": [
            "Number of rows",
            "Duplicate rows",
            "Missing values",
        ],
        "raw": [
            len(raw),
            raw.duplicated().sum(),
            raw.isna().sum().sum(),
        ],
        "cleaned": [
            len(cleaned),
            cleaned.duplicated().sum(),
            cleaned.isna().sum().sum(),
        ],
    }
)

quality_comparison

,metric,raw,cleaned
0,Number of rows,1396,1390
1,Duplicate rows,6,0
2,Missing values,52,0


In [7]:
missing_comparison = pd.DataFrame(
    {
        "raw_missing": raw.isna().sum(),
        "cleaned_missing": cleaned.isna().sum(),
    }
)

missing_comparison[
    (missing_comparison["raw_missing"] > 0)
    | (missing_comparison["cleaned_missing"] > 0)
]

,raw_missing,cleaned_missing
workers,13,0
fuel_consumption_l,13,0
material_delivered_t,13,0
weather,13,0


In [8]:
print("Raw date dtype:", raw["date"].dtype)
print("Cleaned date dtype:", cleaned["date"].dtype)

Raw date dtype: str
Cleaned date dtype: datetime64[us]


In [9]:
print("Raw workers dtype:", raw["workers"].dtype)
print("Cleaned workers dtype:", cleaned["workers"].dtype)

Raw workers dtype: float64
Cleaned workers dtype: int64


In [10]:
validation_results = {
    "workers_positive":
        cleaned["workers"].gt(0).all(),

    "equipment_hours_non_negative":
        cleaned["equipment_hours"].ge(0).all(),

    "fuel_non_negative":
        cleaned["fuel_consumption_l"].ge(0).all(),

    "material_used_non_negative":
        cleaned["material_used_t"].ge(0).all(),

    "material_delivered_non_negative":
        cleaned["material_delivered_t"].ge(0).all(),

    "planned_progress_valid":
        cleaned["planned_progress_pct"]
        .between(0, 100)
        .all(),

    "actual_progress_valid":
        cleaned["actual_progress_pct"]
        .between(0, 100)
        .all(),

    "delay_hours_non_negative":
        cleaned["delay_hours"].ge(0).all(),
}

pd.Series(validation_results)

workers_positive                   True
equipment_hours_non_negative       True
fuel_non_negative                  True
material_used_non_negative         True
material_delivered_non_negative    True
planned_progress_valid             True
actual_progress_valid              True
delay_hours_non_negative           True
dtype: bool

In [11]:
flag_columns = [
    "material_shortage_flag",
    "equipment_breakdown_flag",
    "fuel_anomaly_flag",
]

for column in flag_columns:
    print(
        column,
        sorted(cleaned[column].unique())
    )

material_shortage_flag [np.int64(0), np.int64(1)]
equipment_breakdown_flag [np.int64(0), np.int64(1)]
fuel_anomaly_flag [np.int64(0), np.int64(1)]


In [12]:
event_comparison = pd.DataFrame(
    {
        "event": [
            "Fuel anomaly",
            "Material shortage",
            "Equipment breakdown",
        ],
        "raw": [
            raw["fuel_anomaly_flag"].sum(),
            raw["material_shortage_flag"].sum(),
            raw["equipment_breakdown_flag"].sum(),
        ],
        "cleaned": [
            cleaned["fuel_anomaly_flag"].sum(),
            cleaned["material_shortage_flag"].sum(),
            cleaned["equipment_breakdown_flag"].sum(),
        ],
    }
)

event_comparison

,event,raw,cleaned
0,Fuel anomaly,47,46
1,Material shortage,87,87
2,Equipment breakdown,65,64


In [13]:
cleaned.head()

,date,project_id,workers,equipment_hours,fuel_consumption_l,material_delivered_t,material_used_t,planned_progress_pct,actual_progress_pct,daily_cost_usd,weather,delay_hours,material_shortage_flag,equipment_breakdown_flag,fuel_anomaly_flag
0,2026-01-05,PRJ001,81,31.73,487.87,9.91,8.81,0.42,0.40,7293.92,Cloudy,0.27,0,0,0
1,2026-01-06,PRJ001,86,21.40,297.85,6.57,6.53,0.83,0.70,7063.81,Cloudy,2.46,0,1,0
2,2026-01-07,PRJ001,91,21.77,332.89,5.97,5.62,1.25,0.99,7688.51,Cloudy,3.52,0,1,0
3,2026-01-08,PRJ001,79,25.56,383.14,1.87,3.54,1.67,1.16,6807.22,Light Rain,4.64,1,0,0
4,2026-01-09,PRJ001,88,16.03,261.29,6.75,5.15,2.08,1.42,6222.29,Heavy Rain,4.42,0,0,0


## Cleaning Results

The SitePulse cleaning pipeline successfully transformed the raw operational dataset into a validated analysis-ready dataset.

### Key outcomes

- exact duplicate records were removed
- missing workforce values were imputed using project-level workforce medians
- missing fuel consumption was estimated using equipment operating hours and normal project-level fuel consumption rates
- missing material delivery values were estimated using project and shortage-specific delivery behaviour
- missing weather values were imputed using project/month weather patterns
- operational dates were converted into a datetime-compatible format
- core numerical and business constraints were validated
- binary operational event flags were preserved

The cleaned dataset contains no missing values or exact duplicate records and satisfies the defined data-quality rules.

The processed dataset is now ready for **Exploratory Data Analysis (EDA)** and subsequent feature engineering.